In [ ]:
COMPANY = "pinecone"
BOARD = "ashby"

In [ ]:
import sqlite3, json, bronze, silver
from collections import defaultdict

bronze_db = bronze.open_db(BOARD)
silver_db = silver.open_db()

rows = bronze_db.execute(
    "SELECT * FROM snapshots WHERE company=? ORDER BY source_date",
    (COMPANY,)
).fetchall()
print(f"Bronze records for {COMPANY}: {len(rows)}")

# Group by job_id → {page_type: row}
by_job = defaultdict(dict)
for row in rows:
    by_job[row["job_id"]][row["page_type"]] = row

print(f"Unique jobs: {len(by_job)}")


In [ ]:
from classify import (
    extract_salary_block_from_html, parse_salary_text,
    classify_seniority, classify_work_mode, normalize_department,
    SalaryParseResult,
)
import html2text
import re

def _md(html_text):
    if not html_text:
        return ""
    h = html2text.HTML2Text()
    h.body_width = 0
    h.ignore_images = True
    h.ignore_links = True
    return h.handle(html_text).strip()

def _title_from_html(html_text):
    if not html_text:
        return ""
    m = re.search(r"<title>([^<]+)</title>", html_text, re.IGNORECASE)
    return m.group(1).strip() if m else ""

def merge_to_silver(job_id, page_rows):
    """Merge bronze page_type rows into one silver record.
    api_board → base fields + department_raw
    application → salary
    job_page → description_md (fallback title/location)
    """
    api = page_rows.get("api_board")
    app = page_rows.get("application")
    page = page_rows.get("job_page")
    source_row = api or app or page

    record = {
        "company": source_row["company"],
        "board": source_row["board"],
        "job_id": job_id,
        "source_date": source_row["source_date"],
        "bronze_id": source_row["id"],
    }

    # Base fields from api_board (most structured)
    if api:
        try:
            job = json.loads(api["content"] or "{}")
        except Exception:
            job = {}
        record["title"] = job.get("title", "")
        record["location"] = job.get("location", "")
        record["url"] = job.get("jobUrl") or job.get("absolute_url", "")
        record["department_raw"] = (
            job.get("department") or job.get("team") or
            (job.get("departments") or [{}])[0].get("name", "")
        )

    # Salary from application (JSON-LD)
    if app:
        try:
            jld = json.loads(app["content"] or "{}")
        except Exception:
            jld = {}
        if jld.get("salary_min"):
            record["salary_min"] = int(jld["salary_min"])
            record["salary_max"] = int(jld["salary_max"]) if jld.get("salary_max") else None
            record["currency"] = jld.get("currency", "")
            record["salary_unit"] = jld.get("salary_unit", "")
            record["salary_text"] = jld.get("salary_text", "")
        if not record.get("title"):
            record["title"] = jld.get("title", "")
        if not record.get("location"):
            record["location"] = jld.get("location", "")

    # Description + salary fallback from job_page HTML
    if page:
        content = page["content"] or ""
        record["description_md"] = _md(content)
        if not record.get("salary_min"):
            sal_block = extract_salary_block_from_html(content)
            parsed = parse_salary_text(sal_block) if sal_block else SalaryParseResult("", None, None, None, None)
            if parsed.salary_min:
                record["salary_min"] = parsed.salary_min
                record["salary_max"] = parsed.salary_max
                record["currency"] = parsed.currency or ""
                record["salary_unit"] = parsed.salary_unit or ""
                record["salary_text"] = parsed.salary_text or ""
        if not record.get("title"):
            record["title"] = _title_from_html(content)

    title = record.get("title", "")
    record["seniority"] = classify_seniority(title)
    record["department"] = normalize_department(record.get("department_raw", ""))
    record["work_mode"] = classify_work_mode(record.get("location", ""))
    return record


In [ ]:
processed = upserted = rejected = 0

for job_id, page_rows in by_job.items():
    record = merge_to_silver(job_id, page_rows)
    processed += 1
    ok = silver.upsert_job(silver_db, record)
    upserted += ok
    rejected += not ok

silver.log_run(silver_db, COMPANY, BOARD, processed, upserted, rejected)
print(f"Processed: {processed}  Upserted: {upserted}  Rejected: {rejected}")
